In [83]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    confusion_matrix
)

import joblib

from pathlib import Path

df = pd.read_csv("../data/student_mental_health.csv")

X = df.drop(columns=['Mental_Health_Risk', 'Student_ID'])
y = df['Mental_Health_Risk']



In [84]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

# --- Split 80/20 ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# --- Transformer custom : IQRClipper ---
class IQRClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        Q1 = np.percentile(X, 25, axis=0)
        Q3 = np.percentile(X, 75, axis=0)
        IQR = Q3 - Q1
        self.lower_ = Q1 - 1.5 * IQR
        self.upper_ = Q3 + 1.5 * IQR
        return self
    def transform(self, X):
        return np.clip(X, self.lower_, self.upper_)

# --- Colonnes ---
numerical_cols = [
    'Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week',
    'GPA', 'Social_Media_Hours', 'Screen_Time_Hours',
    'Financial_Stress', 'Family_Support', 'Attendance_Rate'
]
categorical_cols = [
    'Gender', 'Physical_Activity', 'Part_Time_Job',
    'Diet_Quality', 'Previous_Mental_Health_History'
]

# --- Pipeline complet ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('outliers', IQRClipper()),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

# --- Fit sur Train, Transform sur Train et Test ---
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

# --- Sauvegarder les données prétraitées ---
processed_data = {
    'X_train': X_train_processed,
    'X_test':  X_test_processed,
    'y_train': y_train,
    'y_test':  y_test
}
joblib.dump(processed_data, 'preprocessed_data.pkl')
print('Données prétraitées sauvegardées avec succès !')
print('X_train_processed shape:', X_train_processed.shape)
print('X_test_processed shape: ', X_test_processed.shape)

Données prétraitées sauvegardées avec succès !
X_train_processed shape: (1600, 18)
X_test_processed shape:  (400, 18)


In [85]:
import joblib
from pathlib import Path

Path("../models").mkdir(parents=True, exist_ok=True)

processed_data = {
    "X_train": X_train_processed,
    "X_test": X_test_processed,
    "y_train": y_train,
    "y_test": y_test
}

joblib.dump(
    processed_data,
    "../models/preprocessed_data.pkl"
)

['../models/preprocessed_data.pkl']

In [86]:
%pip install xgboost
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier

processed_data = joblib.load(
    "../models/preprocessed_data.pkl"
)

X_train = processed_data["X_train"]
X_test = processed_data["X_test"]

y_train = processed_data["y_train"]
y_test = processed_data["y_test"]

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)

print("\nDistribution de la cible :")
print(y_train.value_counts())

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
X_train : (1600, 18)
X_test : (400, 18)

Distribution de la cible :
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64


In [87]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Classes :", label_encoder.classes_)

Classes : ['High' 'Low' 'Medium']


In [88]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_train_encoded
)

print("Training :", X_train_final.shape)
print("Validation :", X_val.shape)

Training : (1280, 18)
Validation : (320, 18)


In [89]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1
)

In [90]:
from sklearn.preprocessing import OneHotEncoder

# Identifier les colonnes catégorielles
categorical_columns = X_train_final.select_dtypes(
    include=["object", "string"]
).columns.tolist()

# Créer l'encodeur
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Apprendre sur Training uniquement
X_train_encoded = encoder.fit_transform(
    X_train_final[categorical_columns]
)

# Transformer Validation
X_val_encoded = encoder.transform(
    X_val[categorical_columns]
)

# Récupérer les noms des nouvelles colonnes
encoded_columns = encoder.get_feature_names_out(
    categorical_columns
)

# Transformer en DataFrame
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_columns,
    index=X_train_final.index
)

X_val_encoded = pd.DataFrame(
    X_val_encoded,
    columns=encoded_columns,
    index=X_val.index
)

# Supprimer les anciennes colonnes catégorielles
X_train_final = X_train_final.drop(columns=categorical_columns)
X_val = X_val.drop(columns=categorical_columns)

# Combiner les colonnes numériques et encodées
X_train_final = pd.concat(
    [X_train_final, X_train_encoded],
    axis=1
)

X_val = pd.concat(
    [X_val, X_val_encoded],
    axis=1
)

print("Training :", X_train_final.shape)
print("Validation :", X_val.shape)

print("\nTypes des colonnes :")
print(X_train_final.dtypes)

AttributeError: 'numpy.ndarray' object has no attribute 'select_dtypes'

In [ ]:
print("Colonnes non numériques :")

print(
    X_train_final.select_dtypes(
        exclude=["number", "bool"]
    ).columns.tolist()
)

In [ ]:
xgb_model.fit(
    X_train_final,
    y_train_final,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Training terminé !")
print("Meilleur nombre d'arbres :", xgb_model.best_iteration)

In [ ]:
y_val_pred = xgb_model.predict(X_val)

print("Accuracy :", accuracy_score(y_val, y_val_pred))

print("Precision :", precision_score(
    y_val, y_val_pred, average="macro"
))

print("Recall :", recall_score(
    y_val, y_val_pred, average="macro"
))

print("F1-score :", f1_score(
    y_val, y_val_pred, average="macro"
))

print("\nClassification Report :")

print(classification_report(
    y_val,
    y_val_pred,
    target_names=label_encoder.classes_
))

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.title("XGBoost - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

2/ SVM model

In [ ]:
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    random_state=42
)